# FreshGuard v2.1 - 03 Train Detector

Trains YOLO26n as a 1-class `produce` detector with Open Images negatives included in the detector data. Output artifact: `/kaggle/working/freshguard_v2_detector_artifacts/yolo26n_produce_v2_1.pt`.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from ultralytics import YOLO
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.4.0,<9.0.0'], check=True)
    from ultralytics import YOLO

import torch

KAGGLE_INPUT = Path('/kaggle/input')
DATA_DIR = Path(os.environ.get('FRESHGUARD_V2_DETECTOR_DATA_DIR', '/kaggle/input/freshguard-v2-detector-data'))
DATA_YAML = DATA_DIR / 'produce.yaml'
if not DATA_YAML.exists():
    produce_yamls = sorted(KAGGLE_INPUT.rglob('produce.yaml'))
    DATA_YAML = produce_yamls[0] if produce_yamls else DATA_YAML
    DATA_DIR = DATA_YAML.parent
RUNTIME_DATA_YAML = Path('/kaggle/working/produce_runtime.yaml')
yaml_lines = DATA_YAML.read_text().splitlines()
yaml_lines = [f'path: {DATA_DIR}' if line.strip().startswith('path:') else line for line in yaml_lines]
RUNTIME_DATA_YAML.write_text('\n'.join(yaml_lines) + '\n')
OUT_DIR = Path('/kaggle/working/freshguard_v2_detector_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_YAML.exists():
    raise FileNotFoundError(f'Missing YOLO data YAML: {DATA_YAML}')

default_device = '0,1' if torch.cuda.device_count() >= 2 else ('0' if torch.cuda.is_available() else 'cpu')
YOLO_DEVICE = os.environ.get('YOLO_DEVICE', default_device)
default_batch = '16' if ',' in YOLO_DEVICE else '-1'
YOLO_BATCH = int(os.environ.get('YOLO_BATCH', default_batch))
print({'data_yaml': str(DATA_YAML), 'runtime_data_yaml': str(RUNTIME_DATA_YAML), 'out_dir': str(OUT_DIR), 'cuda_devices': torch.cuda.device_count(), 'yolo_device': YOLO_DEVICE, 'yolo_batch': YOLO_BATCH})

In [ ]:
model = YOLO('yolo26n.pt')
results = model.train(
    data=str(RUNTIME_DATA_YAML),
    imgsz=1024,
    epochs=int(os.environ.get('YOLO_EPOCHS', '60')),
    batch=YOLO_BATCH,
    device=YOLO_DEVICE,
    workers=4,
    project='/kaggle/working/freshguard_v2_yolo_runs',
    name='yolo26n_produce_v2_1',
    exist_ok=True,
    patience=12,
    close_mosaic=10,
    cache=False,
    seed=42,
)
print(results)


In [ ]:
run_dir = Path('/kaggle/working/freshguard_v2_yolo_runs/yolo26n_produce_v2_1')
best = run_dir / 'weights' / 'best.pt'
last = run_dir / 'weights' / 'last.pt'
source = best if best.exists() else last
if not source.exists():
    raise FileNotFoundError('YOLO training finished without best.pt or last.pt')

artifact = OUT_DIR / 'yolo26n_produce_v2_1.pt'
shutil.copy2(source, artifact)
metrics = model.val(data=str(RUNTIME_DATA_YAML), imgsz=1024, split='test')
summary = {
    'artifact': str(artifact),
    'size_bytes': artifact.stat().st_size,
    'box_map50': float(metrics.box.map50),
    'box_map': float(metrics.box.map),
}
(OUT_DIR / 'detector_eval_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)
